In [4]:
import numpy as np, pandas as pd, geopandas as gpd
from shapely.geometry import LineString
from shapely.ops import unary_union
from pathlib import Path

DATA = Path("../data")
RESULTS = Path("../results")
RESULTS.mkdir(exist_ok=True)

In [8]:
shore = gpd.read_file(RESULTS / "shorelines.geojson")
first = shore[shore.year == shore.year.min()]
base = max(first.geometry, key=lambda l: l.length).simplify(30)  # smooth jaggies

spacing, half = 100, 500
rows = []
for i in range(int(base.length // spacing) + 1):
    d = i * spacing
    p = base.interpolate(d)
    a = base.interpolate(max(d - 50, 0))
    b = base.interpolate(min(d + 50, base.length))
    dx, dy = b.x - a.x, b.y - a.y
    L = np.hypot(dx, dy)
    nx, ny = -dy / L, dx / L              # perpendicular direction
    if ny < 0:                            # north = seaward on Java's north coast
        nx, ny = -nx, -ny
    line = LineString([(p.x - nx*half, p.y - ny*half),
                       (p.x + nx*half, p.y + ny*half)])
    rows.append(dict(transect_id=i, x=p.x, y=p.y, nx=nx, ny=ny, geometry=line))

transects = gpd.GeoDataFrame(rows, crs=shore.crs)

In [9]:
year_geom = {y: unary_union(list(g.geometry)) for y, g in shore.groupby("year")}

def signed_dist(t, geom):
    inter = t.geometry.intersection(geom)
    pts = [g for g in getattr(inter, "geoms", [inter]) if g.geom_type == "Point"]
    if not pts:
        return np.nan
    q = min(pts, key=lambda g: np.hypot(g.x - t.x, g.y - t.y))  # nearest crossing
    return (q.x - t.x) * t.nx + (q.y - t.y) * t.ny

dist = [dict(transect_id=t.transect_id, year=y, dist=signed_dist(t, g))
        for t in transects.itertuples() for y, g in year_geom.items()]
dist = pd.DataFrame(dist)

def rate(g):
    g = g.dropna()
    return np.polyfit(g.year, g.dist, 1)[0] if len(g) >= 4 else np.nan

rates = dist.groupby("transect_id").apply(rate).rename("rate")

In [10]:
from shapely.geometry import LineString

def make_transect(base, d, half=500, step=50):
    p = base.interpolate(d)
    a = base.interpolate(max(d - step, 0))
    b = base.interpolate(min(d + step, base.length))
    dx, dy = b.x - a.x, b.y - a.y
    L = np.hypot(dx, dy)
    nx, ny = -dy / L, dx / L
    if ny < 0:
        nx, ny = -nx, -ny
    return LineString([(p.x - nx*half, p.y - ny*half),
                       (p.x + nx*half, p.y + ny*half)])

straight = LineString([(0, 0), (1000, 0)])
print(make_transect(straight, 500))          # expect x = 500 for both ends, y from -500 to 500

diagonal = LineString([(0, 0), (1000, 1000)])
print(make_transect(diagonal, 700))

LINESTRING (500 -500, 500 500)
LINESTRING (848.5281374238571 141.42135623730945, 141.42135623730945 848.5281374238571)
